# ANÁLISIS RESTAURANTE


*Instalar e importar librerías

In [ ]:
#!pip install pandas duckdb numpy
import pandas as pd
import numpy as np
import duckdb

# Leer el archivo


In [8]:
restaurante = pd.read_csv("../data/hotel_restaurant_orders.csv", sep = ',' , encoding="utf-8")
restaurante

,OrderID,CustomerID,CustomerName,OrderDate,RestaurantType,MenuCategory,ItemName,Quantity,UnitPrice,TotalPrice,PaymentMethod,ServerName,TableNumber,DayOfWeek,TimeOfDay,SpecialRequest
0,1001,C512,Christopher Lopez,2025-02-04 21:44,Delivery,Beverage,Green Smoothie,2,19.95,39.90,Mobile Payment,-,-,Tuesday,Dinner,NaN
1,1002,C002,Shaun Hensley,2025-11-03 08:05,Takeaway,Food,Pasta Alfredo,2,17.38,34.76,Card,-,-,Monday,Breakfast,Less sugar
2,1003,C198,Allison Williams,2025-03-22 11:13,Delivery,Beverage,Latte,2,7.44,14.88,Cash,-,-,Saturday,Lunch,Skim milk
3,1004,C360,Andrea Shields,2025-04-15 14:36,Takeaway,Food,Beef Burger,2,4.29,8.58,Mobile Payment,-,-,Tuesday,Lunch,Gluten-free
4,1005,C752,Jenna Long,2025-07-28 14:15,Dine-In,Beverage,Latte,2,8.95,17.90,Card,Alice,19,Monday,Lunch,No cheese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3638,4639,C578,Jason Collier,2025-06-14 13:15,Dine-In,Food,Grilled Chicken,1,12.25,12.25,Cash,Juan,3,Saturday,Lunch,Extra napkins
3639,4640,C578,Jason Collier,2025-08-27 18:50,Delivery,Beverage,Mojito,2,15.65,31.30,Card,-,-,Wednesday,Dinner,Less sugar
3640,4641,C578,Jason Collier,2025-03-25 08:50,Delivery,Beverage,Fresh Orange Juice,1,12.37,12.37,Cash,-,-,Tuesday,Breakfast,NaN
3641,4642,C578,Jason Collier,2025-02-10 20:35,Delivery,Beverage,Cappuccino,1,18.61,18.61,Mobile Payment,-,-,Monday,Dinner,NaN


* Revisión de tabla

In [16]:
restaurante.info()

<class 'pandas.DataFrame'>
RangeIndex: 3643 entries, 0 to 3642
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderID         3643 non-null   int64  
 1   CustomerID      3643 non-null   str    
 2   CustomerName    3643 non-null   str    
 3   OrderDate       3643 non-null   str    
 4   RestaurantType  3643 non-null   str    
 5   MenuCategory    3643 non-null   str    
 6   ItemName        3643 non-null   str    
 7   Quantity        3643 non-null   int64  
 8   UnitPrice       3643 non-null   float64
 9   TotalPrice      3643 non-null   float64
 10  PaymentMethod   3643 non-null   str    
 11  ServerName      3643 non-null   str    
 12  TableNumber     3643 non-null   str    
 13  DayOfWeek       3643 non-null   str    
 14  TimeOfDay       3643 non-null   str    
 15  SpecialRequest  2188 non-null   str    
dtypes: float64(2), int64(2), str(12)
memory usage: 455.5 KB


In [ ]:
restaurante.isnull().sum()

OrderID              0
CustomerID           0
CustomerName         0
OrderDate            0
RestaurantType       0
MenuCategory         0
ItemName             0
Quantity             0
UnitPrice            0
TotalPrice           0
PaymentMethod        0
ServerName           0
TableNumber          0
DayOfWeek            0
TimeOfDay            0
SpecialRequest    1455
dtype: int64

* Conexión SQL


In [29]:
query1 = """
    SELECT 
        DATE_TRUNC('MONTH', CAST(OrderDate AS DATE)) AS mes,
        COUNT(DISTINCT OrderID) AS total_ordenes
    FROM 
        restaurante
    GROUP BY 
        DATE_TRUNC('MONTH', CAST(OrderDate AS DATE))
    ORDER BY
        mes ASC;
"""

In [30]:
duckdb.sql(query1).df()


,mes,total_ordenes
0,2025-01-01,307
1,2025-02-01,312
2,2025-03-01,311
3,2025-04-01,308
4,2025-05-01,298
5,2025-06-01,298
6,2025-07-01,314
7,2025-08-01,296
8,2025-09-01,289
9,2025-10-01,315


Las órdenes se distribuyeron de forma estable durante 2025, con un rango entre 289 y 315 órdenes mensuales. Octubre fue el mes con mayor actividad (315 órdenes) y septiembre el de menor actividad (289 órdenes).


In [31]:
query2 = """
    SELECT
        CustomerID AS cliente_id,
        COUNT(DISTINCT OrderID) AS cantidad_ordenes
    FROM restaurante
    GROUP BY cliente_id
    ORDER BY cantidad_ordenes DESC;
"""

In [32]:
duckdb.sql(query2).df()


,cliente_id,cantidad_ordenes
0,C230,42
1,C241,41
2,C243,41
3,C147,39
4,C201,39
...,...,...
475,C840,1
476,C905,1
477,C646,1
478,C728,1


* Registros únicos


In [33]:
query3 = """
    SELECT
        COUNT(*)
    FROM restaurante;
"""

duckdb.sql(query3).df()


,count_star()
0,3643


* Revisión de duplicados

In [34]:
query4 = """
WITH duplicados AS 
    (SELECT
        DISTINCT * 
    FROM restaurante
)

    SELECT
        COUNT(*)
    FROM duplicados;

"""

duckdb.sql(query4).df()

,count_star()
0,3643


In [ ]:
query5 = """
SELECT MenuCategory,
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY MenuCategory
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query5).df()

,MenuCategory,cantidad_ordenes
0,Beverage,1914
1,Food,1729


La categoría de bebidas presenta el mayor número de órdenes, con 1.914 pedidos, seguida por alimentos con 1.729. La diferencia es moderada, por lo que ambas categorías tienen una participación relevante en las ventas.
 

In [35]:
query6 = """
SELECT RestaurantType, 
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY RestaurantType
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query6).df()

,RestaurantType,cantidad_ordenes
0,Dine-In,1757
1,Delivery,1121
2,Takeaway,765


El consumo dentro del establecimiento predomina con 1.757 órdenes. Los domicilios ocupan el segundo lugar con 1.121 órdenes y los pedidos para llevar suman 765, por lo que la experiencia presencial continúa siendo el principal canal de consumo.



In [36]:
query7 = """
SELECT ItemName, 
    COUNT (*) AS cantidad_ordenes 
FROM restaurante
GROUP BY ItemName
ORDER BY cantidad_ordenes DESC;
"""

duckdb.sql(query7).df()

,ItemName,cantidad_ordenes
0,Iced Tea,350
1,Cappuccino,347
2,Margherita Pizza,328
3,Caesar Salad,276
4,Grilled Chicken,275
5,Mojito,272
6,Mineral Water,263
7,Sushi Roll,258
8,Latte,250
9,Fresh Orange Juice,246


Los productos con más órdenes son el té helado (350), el capuchino (347) y la pizza margarita (328). Entre los alimentos también destacan la ensalada César (276) y el pollo a la parrilla (275), lo que evidencia una demanda repartida entre bebidas y opciones de comida.



### Segmentación de clientes según su actividad con el Hotel

En esta sección se segmentan los clientes en tres grupos de valor para el hotel durante el último año. Con las reglas actuales, se obtienen 80 huéspedes VIP, 150 huéspedes conectados y 250 huéspedes esporádicos.

* Huéspedes VIP: han asistido durante 8 o más meses, han realizado 14 o más órdenes y han consumido 300 dólares o más.
* Huéspedes conectados: han asistido durante 4 o más meses, han realizado 6 o más órdenes y han consumido 150 dólares o más, sin cumplir las condiciones de VIP.
* Huéspedes esporádicos: no cumplen simultáneamente las condiciones definidas para los segmentos anteriores. 

In [37]:
query8 = """
WITH agregados AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE_PART('MONTH', CAST(OrderDate AS DATE))) AS recurrencia,
        COUNT(DISTINCT OrderID) AS cantidad_ordenes,
        SUM(TotalPrice) AS valor_total 
    FROM 
        restaurante
    GROUP BY 
        CustomerID
)

SELECT  
    CustomerID,
    CASE
        WHEN recurrencia >=8 AND cantidad_ordenes >=14 AND valor_total >=300 THEN 'Huespedes VIP' 
        WHEN recurrencia >=4 AND cantidad_ordenes >=6 AND valor_total >=150 THEN 'Huespedes conectados' 
        ELSE 'Huespedes esporádicos'
    END AS segmento_valor
FROM 
    agregados
"""


clientes =duckdb.sql(query8).df()
clientes

,CustomerID,segmento_valor
0,C429,Huespedes conectados
1,C628,Huespedes esporádicos
2,C578,Huespedes conectados
3,C186,Huespedes VIP
4,C844,Huespedes esporádicos
...,...,...
475,C1025,Huespedes VIP
476,C1044,Huespedes conectados
477,C1053,Huespedes conectados
478,C1058,Huespedes conectados


In [ ]:
query9 = """

SELECT  
    segmento_valor,
    COUNT(DISTINCT CustomerID) AS cantidad_clientes
FROM 
    clientes
GROUP BY 
    segmento_valor
"""


conteo_clientes =duckdb.sql(query9).df()
conteo_clientes

,segmento_valor,cantidad_clientes
0,Huespedes VIP,80
1,Huespedes conectados,150
2,Huespedes esporádicos,250


* Exportar la base de clientes

In [ ]:
clientes.to_csv("../data/segmentos_clientes.csv", index=False, sep=',', encoding='utf-8')

### Limpiar tabla de datos transaccional 

In [38]:
query10 = """

SELECT  
    OrderID, 
    CustomerID,	
    CAST(OrderDate AS DATE) AS OrderDate,
    RestaurantType,	
    MenuCategory,	
    ItemName,	
    Quantity,	
    UnitPrice,	
    TotalPrice,	
    PaymentMethod,	
    CASE
        WHEN TableNumber = '-' THEN 'N/A'
        ELSE TableNumber
    END AS TableNumber,
    DayOfWeek,	
    TimeOfDay,	
    CASE
        WHEN SpecialRequest IS NULL THEN 'None'
        ELSE SpecialRequest
    END AS SpecialRequest,
    CASE
        WHEN ServerName = '-' THEN 'does not report' 
        ELSE ServerName
    END AS ServerName,

FROM 
    restaurante
"""


transacciones =duckdb.sql(query10).df()
transacciones

,OrderID,CustomerID,OrderDate,RestaurantType,MenuCategory,ItemName,Quantity,UnitPrice,TotalPrice,PaymentMethod,TableNumber,DayOfWeek,TimeOfDay,SpecialRequest,ServerName
0,1001,C512,2025-02-04,Delivery,Beverage,Green Smoothie,2,19.95,39.90,Mobile Payment,N/A,Tuesday,Dinner,None,does not report
1,1002,C002,2025-11-03,Takeaway,Food,Pasta Alfredo,2,17.38,34.76,Card,N/A,Monday,Breakfast,Less sugar,does not report
2,1003,C198,2025-03-22,Delivery,Beverage,Latte,2,7.44,14.88,Cash,N/A,Saturday,Lunch,Skim milk,does not report
3,1004,C360,2025-04-15,Takeaway,Food,Beef Burger,2,4.29,8.58,Mobile Payment,N/A,Tuesday,Lunch,Gluten-free,does not report
4,1005,C752,2025-07-28,Dine-In,Beverage,Latte,2,8.95,17.90,Card,19,Monday,Lunch,No cheese,Alice
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3638,4639,C578,2025-06-14,Dine-In,Food,Grilled Chicken,1,12.25,12.25,Cash,3,Saturday,Lunch,Extra napkins,Juan
3639,4640,C578,2025-08-27,Delivery,Beverage,Mojito,2,15.65,31.30,Card,N/A,Wednesday,Dinner,Less sugar,does not report
3640,4641,C578,2025-03-25,Delivery,Beverage,Fresh Orange Juice,1,12.37,12.37,Cash,N/A,Tuesday,Breakfast,None,does not report
3641,4642,C578,2025-02-10,Delivery,Beverage,Cappuccino,1,18.61,18.61,Mobile Payment,N/A,Monday,Dinner,None,does not report


In [ ]:
transacciones.to_csv("../data/transacciones.csv", index=False, sep=',', encoding='utf-8')

In [44]:
query11 = """
SELECT
    Quantity,
    UnitPrice,
    Quantity * UnitPrice AS valor_total
FROM restaurante
LIMIT 10
"""

duckdb.sql(query11).df()


,Quantity,UnitPrice,valor_total
0,2,19.95,39.90
1,2,17.38,34.76
2,2,7.44,14.88
3,2,4.29,8.58
4,2,8.95,17.90
5,1,4.57,4.57
6,2,10.44,20.88
7,1,8.49,8.49
8,1,15.80,15.80
9,2,17.14,34.28
